# Desenvolvendo uma aplicação para recomendar filmes

* Material de referência: https://www.youtube.com/watch?v=eyEabQRBMQA

In [1]:
import pandas as pd

In [3]:
import zipfile
import requests

In [4]:
url="https://files.grouplens.org/datasets/movielens/ml-25m.zip"

In [5]:
with open("ml-25m.zip", "wb") as f:
    f.write(requests.get(url).content)

In [6]:
import os

with zipfile.ZipFile("ml-25m.zip", "r") as zf:
  zf.extractall("./")

In [7]:
movies = pd.read_csv("./ml-25m/movies.csv")
ratings = pd.read_csv("./ml-25m/ratings.csv")

In [8]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [12]:
import re

# limpando o nome do filme, pois há caracteres que irão atrapalhar
def clean_title(title):
  return re.sub("[^a-zA-Z0-9 ]", "", title)

In [13]:
movies["clean_title"] = movies["title"].apply(clean_title)
movies

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
62418,209157,We (2018),Drama,We 2018
62419,209159,Window of the Soul (2001),Documentary,Window of the Soul 2001
62420,209163,Bad Poems (2018),Comedy|Drama,Bad Poems 2018
62421,209169,A Girl Thing (2001),(no genres listed),A Girl Thing 2001


In [15]:
# transforma títulos em números
from sklearn.feature_extraction.text import TfidfVectorizer

#ngram_range=(1,2): permite buscar além de palavras individuais como "Toy", "Story", "1995"; também buscar coisas como "Toy Story" ou "Story 1995".
vectorizer = TfidfVectorizer(ngram_range=(1,2))

tfidf = vectorizer.fit_transform(movies["clean_title"])

In [23]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# função para buscar baseado nas similaridades
def search(title):
  title = clean_title(title) # pré-processando para que, o título informado corresponda a forma que processamos no dataset

  query_vec = vectorizer.transform([title])

  # calcula as similaridades de todos os titulos com o titulo que foi passado
  similarity = cosine_similarity(query_vec, tfidf).flatten()

  # pegando os mais similares (índices dos 5 mais similares)
  indices = np.argpartition(similarity, -5)[-5:]

  results = movies.iloc[indices]

  return results[::-1] # reverte para o mais similar ficar no topo

In [24]:
# testando o search
search("Avengers")

,movieId,title,genres,clean_title
34536,145676,3 Avengers (1964),(no genres listed),3 Avengers 1964
17067,89745,"Avengers, The (2012)",Action|Adventure|Sci-Fi|IMAX,Avengers The 2012
2063,2153,"Avengers, The (1998)",Action|Adventure,Avengers The 1998
40636,159920,Shaolin Avengers (1994),Action,Shaolin Avengers 1994
45394,170297,Ultimate Avengers 2 (2006),Action|Animation|Sci-Fi,Ultimate Avengers 2 2006


In [27]:
# para colocar um input no jupyter
import ipywidgets as widgets
from IPython.display import display

# cria o input
movie_input = widgets.Text(
    value="Toy Story",
    description="Movie Title:",
    disabled=False
)

movie_list = widgets.Output()

def on_type(data):
  with movie_list:
    movie_list.clear_output()
    title = data["new"]
    if len(title) > 5:
      display(search(title))

# fica escutando, e quando acontecer algo chama "on_type"
movie_input.observe(on_type, names="value")

display(movie_input, movie_list)

Text(value='Toy Story', description='Movie Title:')

Output()

## Second half: Antes encontramos filmes que eram similares ao que digitamos, mas queremos na verdade encontrar filmes que são similares ao filme que digitamos.

In [28]:
# encontrando usuarios que gostaram do mesmo filme
movie_id = 1
similar_users = ratings[(ratings["movieId"] == 1) & (ratings["rating"] >= 4)]["userId"].unique()

In [29]:
# id das pessoas que gostaram do mesmo filme
similar_users

array([    36,     75,     86, ..., 162518, 162519, 162530])

In [30]:
similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] >= 4)]

In [31]:
similar_user_recs

,userId,movieId,rating,timestamp
5101,36,1,5.0,857131378
5104,36,11,4.0,840790432
5105,36,34,5.0,834413787
5106,36,46,4.0,858776254
5108,36,60,4.0,839926627
...,...,...,...,...
24998389,162530,3735,5.0,989808150
24998390,162530,3751,4.0,989808512
24998391,162530,3763,5.0,989809659
24998392,162530,4187,5.0,989809274


In [32]:
similar_user_recs = similar_user_recs["movieId"]

In [ ]:
# filtrar filmes que ao menos 10% dos usuários similares gostaram

In [36]:
similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

similar_user_recs = similar_user_recs[similar_user_recs > .10]

In [37]:
similar_user_recs

,count
movieId,
1,1.000000
260,0.521102
318,0.516733
356,0.502443
296,0.458315
...,...
1358,0.101733
1485,0.101659
16,0.101140


In [38]:
all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]

In [39]:
all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [40]:
all_user_recs

,count
movieId,
318,0.332559
296,0.276637
2571,0.237144
356,0.228624
593,0.219531
...,...
2078,0.014215
1485,0.013025
2080,0.012825


In [42]:
rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
rec_percentages.columns = ["similar", "all"]

In [43]:
rec_percentages

,similar,all
movieId,,
1,1.000000,0.121207
260,0.521102,0.215934
318,0.516733,0.332559
356,0.502443,0.228624
296,0.458315,0.276637
...,...,...
1358,0.101733,0.029705
1485,0.101659,0.013025
16,0.101140,0.033154


In [44]:
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

In [45]:
rec_percentages = rec_percentages.sort_values("score", ascending=False)

In [46]:
rec_percentages

,similar,all,score
movieId,,,
708,0.107878,0.010045,10.739077
2355,0.249889,0.024383,10.248480
2080,0.109211,0.012825,8.515203
1022,0.102473,0.012214,8.389772
1,1.000000,0.121207,8.250332
...,...,...,...
58559,0.201392,0.142334,1.414925
7361,0.140234,0.101297,1.384388
79132,0.174219,0.127675,1.364553


In [47]:
rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")

,similar,all,score,movieId,title,genres,clean_title
693,0.107878,0.010045,10.739077,708,"Truth About Cats & Dogs, The (1996)",Comedy|Romance,Truth About Cats Dogs The 1996
2264,0.249889,0.024383,10.248480,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,Bugs Life A 1998
1991,0.109211,0.012825,8.515203,2080,Lady and the Tramp (1955),Animation|Children|Comedy|Romance,Lady and the Tramp 1955
999,0.102473,0.012214,8.389772,1022,Cinderella (1950),Animation|Children|Fantasy|Musical|Romance,Cinderella 1950
0,1.000000,0.121207,8.250332,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
435,0.135125,0.016558,8.160812,440,Dave (1993),Comedy|Romance,Dave 1993
721,0.169776,0.021410,7.929787,736,Twister (1996),Action|Adventure|Romance|Thriller,Twister 1996
1989,0.112691,0.014215,7.927372,2078,"Jungle Book, The (1967)",Animation|Children|Comedy|Musical,Jungle Book The 1967
1,0.135940,0.017163,7.920634,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
588,0.117355,0.014827,7.915112,596,Pinocchio (1940),Animation|Children|Fantasy|Musical,Pinocchio 1940
